# 🚀 Đồng Bộ & Render Tự Động Toàn Bộ Đại Dự Án 4K 60fps MAX CINEMA (8 Video Lễ Hội Diễu Hành + 30 Phim Discovery) Thẳng Vào Google Drive
Notebook này chạy **100% trên Google Cloud (Colab GPU/CPU)**:
1. Tự động kéo toàn bộ các bản **4K 60fps MAX CINEMA (`45 Mbps`, `CRF 15`, `Lanczos + CAS + 3D Denoise`, `Stereo 48kHz + Nhạc Lễ Hội`)** đã render sẵn từ GitHub Cloud (`dung11122d/cloud-video-studio`), ghép các part `.part00/.part01...` thành file `.mp4` nguyên khối (`3.6 GB – 8.5 GB`) và đẩy thẳng vào đúng 3 thư mục Google Drive của bạn.
2. Nếu bất kỳ dự án Lễ Hội nào (`01..08`) bị Google Drive khóa hạn mức tải ẩn danh (Anonymous Quota) trên GitHub, Colab sẽ **tự động dùng quyền đăng nhập Google Drive chính chủ (`GDRIVE_ACCESS_TOKEN`) + GPU NVIDIA trên Cloud** để tải cảnh gốc với tốc độ 200 MB/s, render chuẩn **4K 60fps MAX CINEMA** và lưu thẳng vào đúng thư mục Google Drive của bạn!

In [ ]:
import os
import re
import json
import shutil
import subprocess
import urllib.request
from collections import defaultdict
import google.auth
from google.auth.transport.requests import Request
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

# 1. Xác thực trực tiếp với Google Drive API trên Cloud
auth.authenticate_user()
creds, _ = google.auth.default()
creds.refresh(Request())
os.environ['GDRIVE_ACCESS_TOKEN'] = creds.token
drive_service = build('drive', 'v3', credentials=creds)
os.makedirs('/content/tmp_sync', exist_ok=True)

def get_or_create_subfolder(parent_id, subfolder_name):
    q = f"'{parent_id}' in parents and name = '{subfolder_name}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
    res = drive_service.files().list(q=q, fields='files(id, name)', supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
    folders = res.get('files', [])
    if folders:
        return folders[0]['id']
    meta = {'name': subfolder_name, 'mimeType': 'application/vnd.google-apps.folder', 'parents': [parent_id]}
    return drive_service.files().create(body=meta, fields='id', supportsAllDrives=True).execute()['id']

def list_existing_files(folder_id):
    existing = {}
    page_token = None
    while True:
        r = drive_service.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields='nextPageToken, files(id, name, size)',
            pageToken=page_token,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True
        ).execute()
        for f in r.get('files', []):
            existing[f['name']] = int(f.get('size', 0))
        page_token = r.get('nextPageToken')
        if not page_token:
            break
    return existing

def upload_to_drive(local_path, filename, target_folder_id, existing):
    fsize = os.path.getsize(local_path)
    if filename in existing and existing[filename] == fsize:
        print(f'⏭️ Đã có trên Drive: {filename} ({fsize/(1024*1024):.1f} MB)')
        return
    mime = 'video/mp4' if filename.endswith('.mp4') else 'application/json'
    media = MediaFileUpload(local_path, mimetype=mime, chunksize=64*1024*1024, resumable=True)
    body = {'name': filename, 'parents': [target_folder_id]}
    request = drive_service.files().create(body=body, media_body=media, fields='id', supportsAllDrives=True)
    response = None
    while response is None:
        status, response = request.next_chunk()
    existing[filename] = fsize
    print(f'✅ Đã đẩy thẳng lên Google Drive: {filename} ({fsize/(1024*1024):.1f} MB)')

def sync_release_to_drive(api_url, route_fn):
    try:
        req = urllib.request.Request(api_url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as resp:
            rel = json.loads(resp.read().decode('utf-8'))
            assets = rel.get('assets', [])
    except Exception as e:
        print(f'⚠️ Chưa tìm thấy release {api_url}: {e}')
        return

    single_assets = [a for a in assets if not re.search(r'\.part\d+$', a['name'])]
    part_groups = defaultdict(list)
    for a in assets:
        m = re.match(r'^(.+\.mp4)\.part(\d+)$', a['name'])
        if m:
            part_groups[m.group(1)].append((int(m.group(2)), a))

    for a in single_assets:
        name = a['name']
        target_id, existing = route_fn(name)
        if name in existing and existing[name] == a['size']:
            print(f'⏭️ Đã có sẵn trên Drive: {name}')
            continue
        tmp_file = f'/content/tmp_sync/{name}'
        print(f'⬇️ Kéo trên Cloud {name} ({a["size"]/(1024*1024):.1f} MB)...')
        urllib.request.urlretrieve(a['browser_download_url'], tmp_file)
        upload_to_drive(tmp_file, name, target_id, existing)
        os.remove(tmp_file)

    for base_name, parts in sorted(part_groups.items()):
        parts.sort(key=lambda x: x[0])
        total_sz = sum(p[1]['size'] for p in parts)
        target_id, existing = route_fn(base_name)
        if base_name in existing and existing[base_name] == total_sz:
            print(f'⏭️ Đã có sẵn bản Max-Bitrate trên Drive: {base_name} ({total_sz/(1024*1024):.1f} MB)')
            continue
        merged_path = f'/content/tmp_sync/{base_name}'
        print(f'🎬 Đang ghép nối {len(parts)} phần của siêu video 4K60 MAX CINEMA {base_name} ({total_sz/(1024*1024):.1f} MB) trên Cloud...')
        with open(merged_path, 'wb') as out_f:
            for p_idx, p_asset in parts:
                p_tmp = f'/content/tmp_sync/{p_asset["name"]}'
                urllib.request.urlretrieve(p_asset['browser_download_url'], p_tmp)
                with open(p_tmp, 'rb') as in_f:
                    while True:
                        buf = in_f.read(64 * 1024 * 1024)
                        if not buf:
                            break
                        out_f.write(buf)
                os.remove(p_tmp)
        upload_to_drive(merged_path, base_name, target_id, existing)
        os.remove(merged_path)

# A. Đồng bộ 8 Dự Án Lễ Hội & Diễu Hành (parade-4k60-max-cinema) vào 2 Folder Lễ Hội
parade_f1_id = get_or_create_subfolder('1RH5Ep5PjAK5G_ZfywDfkssttD_rRQjQ3', 'Rendered_4K60_Parade_MAX_CINEMA')
parade_f2_id = get_or_create_subfolder('1DsS5dXrzw1eNHVl6CXDhlA_PUqTcEQji', 'Rendered_4K60_Parade_MAX_CINEMA')
parade_f1_existing = list_existing_files(parade_f1_id)
parade_f2_existing = list_existing_files(parade_f2_id)

def route_parade(filename):
    if filename.startswith(('07_', '08_')):
        return parade_f2_id, parade_f2_existing
    return parade_f1_id, parade_f1_existing

print('=== 1. ĐỒNG BỘ 8 ĐẠI DỰ ÁN LỄ HỘI & DIỄU HÀNH 4K 60FPS MAX CINEMA TỪ GITHUB CLOUD ===')
sync_release_to_drive('https://api.github.com/repos/dung11122d/cloud-video-studio/releases/tags/parade-4k60-max-cinema', route_parade)

# Tự động kiểm tra và render bổ sung bằng quyền Google Drive API chính chủ trên Colab nếu bất kỳ dự án Lễ Hội nào (1..8) bị kẹt hạn mức ẩn danh
if not os.path.exists('/content/cloud-video-studio'):
    subprocess.run(['git', 'clone', 'https://github.com/dung11122d/cloud-video-studio.git', '/content/cloud-video-studio'], check=True)
subprocess.run(['pip', 'install', '-q', 'opencv-python-headless', 'numpy', 'gdown', 'yt-dlp'], check=False)

has_gpu = subprocess.run(['nvidia-smi'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
if has_gpu:
    os.environ['USE_NVENC'] = '1'
    print('⚡ Phát hiện GPU NVIDIA trên Colab -> Bật tăng tốc phần cứng h264_nvenc 4K 60fps!')

manifest = json.load(open('/content/cloud-video-studio/scripts/parade_manifest.json', encoding='utf-8'))
for p in manifest:
    mp4_name = f"{p['slug']}_4K60_MAX_CINEMA.mp4"
    target_id, existing = route_parade(mp4_name)
    if mp4_name not in existing:
        print(f"\n🚀 Đang render trực tiếp trên Google Cloud (với Token chính chủ không giới hạn): [{p['idx']}] {mp4_name}...")
        creds.refresh(Request())
        os.environ['GDRIVE_ACCESS_TOKEN'] = creds.token
        subprocess.run(['python3', '-u', '/content/cloud-video-studio/scripts/festival_parade_master_pipeline.py', str(p['idx'])], cwd='/content/cloud-video-studio', check=True)
        for out_f in os.listdir('/content/cloud-video-studio/output_parade'):
            full_p = f'/content/cloud-video-studio/output_parade/{out_f}'
            t_id, ex_map = route_parade(out_f)
            upload_to_drive(full_p, out_f, t_id, ex_map)
            os.remove(full_p)

# B. Đồng bộ 30 Bộ Phim Tài Liệu Discovery (discovery-4k60-max-cinema) vào Folder Discovery
disc_id = get_or_create_subfolder('1wZmfSYFBWDwhydvtFGR9eK-GKspjvJlz', 'Rendered_4K60_Discovery_Masters')
disc_existing = list_existing_files(disc_id)
print('\n=== 2. ĐỒNG BỘ 30 PHIM TÀI LIỆU DISCOVERY 4K 60FPS MAX CINEMA ===')
sync_release_to_drive('https://api.github.com/repos/dung11122d/cloud-video-studio/releases/tags/discovery-4k60-max-cinema', lambda fn: (disc_id, disc_existing))

print('🎉 HOÀN TẤT! TOÀN BỘ 8 ĐẠI DỰ ÁN LỄ HỘI & 30 PHIM DISCOVERY 4K 60FPS MAX CINEMA ĐÃ NẰM TRONG GOOGLE DRIVE CỦA BẠN!')